# Partie 1 - Mise en place du cluster et positionnement
Objectif : verifier que le cluster Spark standalone (1 Master + 2 Workers), Kafka et MinIO sont
operationnels, puis ouvrir une `SparkSession` connectee a `spark://spark-master:7077`.

Prerequis (depuis la machine hote) :
```bash
docker compose up -d          # kafka-init cree le topic, minio-init cree le bucket lake
docker compose ps
```

In [2]:
import os, sys, socket, json, urllib.request
sys.path.insert(0, "/home/sahlpay/jobs"); sys.path.insert(0, "../jobs")
from sahlpay_common import get_spark, SPARK_MASTER, KAFKA_BOOTSTRAP, LAKE

spark = get_spark("SahlPay-P1-Cluster", cores_max=4)
sc = spark.sparkContext

print("Version de Spark      :", spark.version)
print("Master                :", sc.master)
print("appName               :", sc.appName)
print("Coeurs disponibles    : sc.defaultParallelism =", sc.defaultParallelism)
print("Application id        :", sc.applicationId)
print("Spark UI (driver)     :", sc.uiWebUrl)

Version de Spark      : 3.5.3
Master                : spark://spark-master:7077
appName               : SahlPay-P1-Cluster
Coeurs disponibles    : sc.defaultParallelism = 4
Application id        : app-20261118210643-0001
Spark UI (driver)     : http://jupyter:4040


### Etat du cluster vu par le Master (API JSON de l'interface :8080)

In [3]:
try:
    state = json.load(urllib.request.urlopen("http://spark-master:8080/json/", timeout=5))
    print("Statut du Master :", state["status"], "| cores totaux :", state["cores"],
          "| memoire totale (Mo) :", state["memory"])
    for w in state["workers"]:
        print(f"  - {w['id']} @ {w['host']}  state={w['state']}  cores={w['cores']}  memoire={w['memory']} Mo")
    print("Workers ALIVE :", sum(1 for w in state["workers"] if w["state"] == "ALIVE"))
except Exception as e:
    print("Interface du Master inaccessible :", e)

Statut du Master : ALIVE | cores totaux : 4 | memoire totale (Mo) : 4096
  - worker-20261118205754-spark-worker-2-45013 @ spark-worker-2  state=ALIVE  cores=2  memoire=2048 Mo
  - worker-20261118205754-spark-worker-1-43731 @ spark-worker-1  state=ALIVE  cores=2  memoire=2048 Mo
Workers ALIVE : 2


### Executors effectivement alloues a cette application

In [4]:
executors = sc._jsc.sc().getExecutorMemoryStatus().keySet().toString()
print("Executors (host:port) :", executors)

# Petit job distribue : sur quel worker chaque partition est-elle traitee ?
def where(_):
    import socket
    yield socket.gethostname()

print(sorted(set(sc.parallelize(range(sc.defaultParallelism * 4), sc.defaultParallelism * 4)
                  .mapPartitions(where).collect())))

Executors (host:port) : Set(spark-worker-2:33769, jupyter:37855, spark-worker-1:41133, spark-worker-2:46473, spark-worker-1:37659)
['spark-worker-1', 'spark-worker-2']


### Verification de MinIO (bucket `lake`) et de Kafka (topic `transactions`)

In [6]:
# Ecriture / lecture d'un petit fichier de controle dans le bucket via S3A
spark.createDataFrame([("ok", spark.version)], ["check", "spark_version"]) \
     .write.mode("overwrite").json(f"{LAKE}/healthcheck")
spark.read.json(f"{LAKE}/healthcheck").show()

from kafka import KafkaAdminClient
admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP)
for t in admin.describe_topics(["transactions"]):
    print("Topic", (t.get("topic") or t.get("name")), "-> partitions :", len(t["partitions"]))
admin.close()

+-----+-------------+
|check|spark_version|
+-----+-------------+
|   ok|        3.5.3|
+-----+-------------+

Topic transactions -> partitions : 3


### Elements pour le rapport (1 page maximum)

**Roles dans notre deploiement**
* **Driver** : le processus Python/JVM du conteneur `jupyter` (mode client). Il execute le
  `main`, construit le DAG, le decoupe en *stages* et en *tasks* et les planifie.
* **Cluster Manager** : le Master standalone (`spark-master:7077`). Il connait les Workers et
  alloue les ressources (coeurs / memoire) demandees par l'application.
* **Workers / Executors** : `spark-worker-1` et `spark-worker-2` (2 coeurs, 2 Go chacun). Pour
  cette application, chaque Worker lance des *Executors* (JVM de 1 coeur / 1 Go, voir la liste
  ci-dessus) ; ils executent les tasks, gardent les donnees en cache et lisent/ecrivent directement
  MinIO via S3A. Les applications streaming et GraphX sont limitees a 2 coeurs
  (`spark.cores.max`) pour pouvoir partager le cluster.
* **SparkContext / SparkSession** : point d'entree cree par le driver ; il se connecte au Master,
  obtient les executors et expose les API RDD (`sc`) et SQL/DataFrame (`spark`).

**Deux limites de MapReduce/Hadoop corrigees par Spark (reliees au projet)**
1. *Ecriture sur disque entre chaque etape* : un job MapReduce materialise le resultat de chaque
   phase sur HDFS. Nos calculs (filtre -> agregation par ville -> top commercants -> jointure) sont
   des chaines d'operations ; Spark les pipeline en memoire dans un DAG et ne fait de shuffle
   qu'aux frontieres de stage.
2. *Batch uniquement, latence elevee* : MapReduce ne traite pas de flux. SahlPay a besoin
   d'indicateurs quasi temps reel (fenetres de 10 min, taux de rejet) : Spark offre le meme
   moteur pour le batch (Parties 2-3), le streaming depuis Kafka (Parties 4-5) et les graphes
   (GraphX / GraphFrames, Partie 6), sans changer d'outil.

In [7]:
spark.stop()